# 02 - Data Preparation and Features

Create the modeling frames used for the short-term and medium-term forecasts.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from group5_energy.pipeline import (
    HALF_TARGET,
    DAILY_TARGET,
    add_history_lags,
    latest_clients,
    load_daily_history,
    load_daily_weather,
    load_half_hourly_history,
    load_holidays,
    load_hourly_weather,
    load_temperatures,
    prepare_daily_frame,
    prepare_half_hourly_frame,
)


In [ ]:
half_history = load_half_hourly_history()
daily_history = load_daily_history()
weather_hourly = load_hourly_weather()
temperatures = load_temperatures()
weather_daily = load_daily_weather()
holidays = load_holidays()

half_clients = latest_clients(half_history, "DateTime")
daily_clients = latest_clients(daily_history, "Date")

half_features = prepare_half_hourly_frame(half_history, weather_hourly, temperatures, holidays, half_clients)
daily_features = prepare_daily_frame(daily_history, weather_daily, holidays, daily_clients)
half_features = add_history_lags(half_features, HALF_TARGET, "half_hourly")
daily_features = add_history_lags(daily_features, DAILY_TARGET, "daily")

half_features.shape, daily_features.shape

In [ ]:
half_features[[
    "Acorn", "DateTime", "Conso_moy", "temperature", "temperature_half_hour",
    "is_holiday", "half_hour_slot", "lag_48", "lag_336", "rolling_48_mean"
]].head(10)

In [ ]:
daily_features[[
    "Acorn", "Date", "Conso_kWh", "temperatureMean", "is_holiday",
    "weekday", "lag_1", "lag_7", "rolling_7_mean"
]].head(10)

In [ ]:
missing_half = half_features.isna().mean().sort_values(ascending=False).head(15)
missing_daily = daily_features.isna().mean().sort_values(ascending=False).head(15)
pd.DataFrame({"half_hourly_missing_rate": missing_half}).join(
    pd.DataFrame({"daily_missing_rate": missing_daily}), how="outer"
)

The first lag rows have missing values by construction. The model pipeline imputes these values during training and prediction.